# 03 - Analysis & Modeling

Statistical analysis, anomaly detection, and predictive modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

BASE_DIR = Path('..')
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
FIGURES_DIR = BASE_DIR / 'outputs' / 'figures'

df_enrol = pd.read_parquet(PROCESSED_DIR / 'enrolment_cleaned.parquet')
df_bio = pd.read_parquet(PROCESSED_DIR / 'biometric_cleaned.parquet')
df_demo = pd.read_parquet(PROCESSED_DIR / 'demographic_cleaned.parquet')

print('Data loaded successfully')

## 3.1 Anomaly Detection

In [ ]:
# Identify anomalous pincodes using IQR method
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return df[(df[column] < lower) | (df[column] > upper)]

outliers_enrol = detect_outliers_iqr(df_enrol, 'total_enrol')
print(f"Enrolment outliers: {len(outliers_enrol):,} records ({100*len(outliers_enrol)/len(df_enrol):.2f}%)")
print(f"Top outlier pincodes:")
display(outliers_enrol.groupby('pincode')['total_enrol'].sum().sort_values(ascending=False).head(10))

In [ ]:
# Isolation Forest for multivariate anomaly detection
features = ['age_0_5', 'age_5_17', 'age_18_greater']
X = df_enrol[features].values

iso_forest = IsolationForest(contamination=0.05, random_state=42, n_jobs=-1)
df_enrol['anomaly'] = iso_forest.fit_predict(X)

anomalies = df_enrol[df_enrol['anomaly'] == -1]
print(f"Isolation Forest anomalies: {len(anomalies):,} records")

# Anomaly distribution by state
anomaly_by_state = anomalies.groupby('state').size().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 6))
anomaly_by_state.plot(kind='bar', ax=ax, color='coral')
ax.set_title('Top 10 States by Anomalous Records', fontsize=14, fontweight='bold')
ax.set_xlabel('State')
ax.set_ylabel('Anomaly Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '11_anomalies_by_state.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.2 Regional Disparity Analysis

In [ ]:
# Identify underserved districts (low activity)
district_activity = df_enrol.groupby(['state', 'district']).agg({
    'total_enrol': 'sum',
    'pincode': 'nunique',
    'date': 'nunique'
}).rename(columns={'pincode': 'pincodes', 'date': 'active_days'})

# Per-pincode-day activity
district_activity['avg_daily_per_pincode'] = district_activity['total_enrol'] / (district_activity['pincodes'] * district_activity['active_days'])

# Bottom 20 districts by activity
bottom_districts = district_activity['avg_daily_per_pincode'].sort_values().head(20)
print("Underserved Districts (lowest avg daily enrolments per pincode):")
display(bottom_districts)

In [ ]:
# Visualize disparity
fig, ax = plt.subplots(figsize=(12, 8))
labels = [f"{idx[1][:15]} ({idx[0][:3]})" for idx in bottom_districts.index]
ax.barh(labels[::-1], bottom_districts.values[::-1], color=sns.color_palette('Reds_r', 20))
ax.set_xlabel('Avg Daily Enrolments per Pincode')
ax.set_title('20 Most Underserved Districts', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '12_underserved_districts.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.3 Trend Forecasting

In [ ]:
# Simple linear regression forecast
daily_total = df_enrol.groupby('date')['total_enrol'].sum().reset_index()
daily_total['day_num'] = (daily_total['date'] - daily_total['date'].min()).dt.days

X = daily_total[['day_num']].values
y = daily_total['total_enrol'].values

model = LinearRegression()
model.fit(X, y)
daily_total['predicted'] = model.predict(X)

# Forecast next 30 days
future_days = np.arange(X.max() + 1, X.max() + 31).reshape(-1, 1)
future_predictions = model.predict(future_days)

print(f"Trend coefficient: {model.coef_[0]:,.2f} enrolments/day")
print(f"Next 30-day forecast total: {future_predictions.sum():,.0f}")

In [ ]:
# Visualize trend
fig, ax = plt.subplots(figsize=(14, 6))
ax.scatter(daily_total['date'], daily_total['total_enrol'], alpha=0.5, label='Actual', s=20)
ax.plot(daily_total['date'], daily_total['predicted'], color='red', linewidth=2, label='Trend')

# Add forecast
last_date = daily_total['date'].max()
future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=30)
ax.plot(future_dates, future_predictions, color='orange', linestyle='--', linewidth=2, label='Forecast')

ax.set_xlabel('Date')
ax.set_ylabel('Daily Enrolments')
ax.set_title('Enrolment Trend & 30-Day Forecast', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / '13_trend_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.4 Key Findings Summary

In [ ]:
findings = {
    'Total Enrolments': f"{df_enrol['total_enrol'].sum():,.0f}",
    'Anomalous Records': f"{len(anomalies):,} ({100*len(anomalies)/len(df_enrol):.1f}%)",
    'Underserved Districts': len(bottom_districts),
    'Trend Direction': 'Increasing' if model.coef_[0] > 0 else 'Decreasing',
    'Daily Growth Rate': f"{model.coef_[0]:,.0f} enrolments/day",
    '30-Day Forecast': f"{future_predictions.sum():,.0f} projected enrolments"
}

print("\n=== Key Analytical Findings ===")
for k, v in findings.items():
    print(f"{k}: {v}")

In [ ]:
# Save findings for report
findings_df = pd.DataFrame(list(findings.items()), columns=['Metric', 'Value'])
findings_df.to_csv(BASE_DIR / 'outputs' / 'key_findings.csv', index=False)
print("\n✅ Analysis complete. Findings saved.")